In [ ]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [19]:
from datasets import load_dataset

In [ ]:
# https://huggingface.co/datasets/larryvrh/Chinese-Poems

data = load_dataset("larryvrh/Chinese-Poems")

In [ ]:
data = data['train']

In [ ]:
data['train'][0]

In [5]:
from transformers import BertTokenizer, GPT2LMHeadModel

In [ ]:
tokenizer = BertTokenizer.from_pretrained("uer/gpt2-distil-chinese-cluecorpussmall")

In [7]:
def tokenize_fn(text):
    return tokenizer(text, padding="max_length", truncation=True, max_length=512)

In [8]:
def datapipe(data):
    outputs = tokenize_fn(data["content"])

    input_ids = outputs["input_ids"]
    attention_mask = outputs["attention_mask"]

    labels = input_ids.copy()

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }

In [9]:
tokenized_data = data.with_transform(datapipe)

In [ ]:
print(tokenized_data['train'][0])

In [11]:
from transformers import DataCollatorForLanguageModeling

In [12]:
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
print(collator)

In [14]:
model = GPT2LMHeadModel.from_pretrained("uer/gpt2-distil-chinese-cluecorpussmall")

In [15]:
from transformers import Trainer, TrainingArguments

In [16]:
training_args = TrainingArguments(
    output_dir="./chinese-poems-gpt2-finetuned",
    eval_strategy='epoch',
    per_device_train_batch_size=2,
    num_train_epochs = 10,
    remove_unused_columns=False,
    learning_rate=1e-4,
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=tokenized_data["train"],
)

In [ ]:
trainer.train()